# Cox process
Want to reproduce [this](https://tinygp.readthedocs.io/en/stable/tutorials/likelihoods.html) using BlackJAX only, no numpyro.

## BlackJAX version

In [ ]:
import datetime

start = datetime.datetime.now()

import matplotlib.pyplot as plt
import numpy as np
from typing import NamedTuple
from functools import partial
import jax
import jax.numpy as jnp
import blackjax
from jaxtyping import Float
from tinygp import GaussianProcess, kernels

jax.config.update('jax_num_cpu_devices', 8)
jax.config.update("jax_enable_x64", True)

rnd_state = int(datetime.date.today().strftime("%Y%m%d"))
rng_key = jax.random.key(rnd_state)
SIGMA = 5  # TODO check
T = 20  # nb of datapoints
BURNIN = 5_000

print(start.strftime("%Y-%m-%d %H:%M:%S"))
jax.devices()

In [ ]:
random = np.random.default_rng(203618)
x = np.linspace(-3, 3, T)
true_log_rate = 2 * np.cos(2 * x)
y = random.poisson(np.exp(true_log_rate))

In [ ]:
class Params(NamedTuple):
    u: Float[jax.Array, ""]
    m: Float[jax.Array, "{T}"]


def init_param(key, t):
    key1, key2 = jax.random.split(key, 2)
    u = jax.random.uniform(key1)

    kernel = SIGMA**2 * kernels.Matern52(jnp.exp(u))
    gp = GaussianProcess(kernel, t, diag=1e-5)
    # array of shape T
    m = gp.sample(key2)

    return Params(
        u=u,
        m=m,
    )


rng_key, keys_init = jax.random.split(rng_key)
initial_params = init_param(keys_init, x)
initial_params

In [ ]:
@jax.jit
def logdensity(t, y, params):
    # hyperprior
    log_hyperprior = jax.scipy.stats.norm.logpdf(params.u)

    # (latent) prior
    kernel = SIGMA**2 * kernels.Matern52(params.u)
    gp = GaussianProcess(kernel, x, diag=1e-5)
    log_prior = gp.log_probability(params.m)

    # likelihood
    log_likelihood = jnp.sum(jax.scipy.stats.poisson.logpmf(y, jnp.exp(params.m)))

    return log_hyperprior + log_prior + log_likelihood


def inference_loop_multiple_chains(
    kernel, rng_key, initial_states, tuned_params, log_prob_fn, num_samples, num_chains
):

    def step_fn(key, state, **params):
        return kernel(key, state, log_prob_fn, **params)

    def one_step(states, rng_key):
        keys = jax.random.split(rng_key, num_chains)
        states, infos = jax.vmap(step_fn)(keys, states, **tuned_params)
        return states, (states, infos)

    keys = jax.random.split(rng_key, num_samples)
    _, (states, infos) = jax.lax.scan(one_step, initial_states, keys)

    return (states, infos)

In [ ]:
inv_mass_matrix = np.ones(T + 1) * 0.1
step_size = 1e-3

initial_position = blackjax.nuts(
    partial(logdensity, x, y), step_size, inv_mass_matrix
).init(initial_params)
assert initial_position.logdensity == logdensity(x, y, initial_params)  # THIS
print(
    f"Log density of {initial_position.logdensity:.2f} at initial position {initial_position}"
)

Insted of a random inverse matrix, use a indow adaptation as described [here](https://blackjax-devs.github.io/blackjax/examples/quickstart.html#use-stan-s-window-adaptation) to find inverse mass matrix.

In [ ]:
warmup = blackjax.window_adaptation(blackjax.nuts, partial(logdensity, x, y))
rng_key, warmup_key, sample_key = jax.random.split(rng_key, 3)
(state, parameters_nuts), _ = warmup.run(warmup_key, initial_params, num_steps=10_000)
parameters_nuts

In [ ]:
def inference_loop(rng_key, kernel, initial_state, num_samples):
    @jax.jit
    def one_step(state, rng_key):
        state, _ = kernel(rng_key, state)
        return state, state

    keys = jax.random.split(rng_key, num_samples)
    _, states = jax.lax.scan(one_step, initial_state, keys)

    return states

In [ ]:
%%time

# use obtained parameters to define a new kernel
kernel = jax.jit(blackjax.nuts(partial(logdensity, x, y), **parameters_nuts).step)
states = inference_loop(sample_key, kernel, state, 1_000 + BURNIN)
m_samples = states.position.m[BURNIN:]
m_samples.shape

In [ ]:
# diagnostics see https://arviz-devs.github.io/EABM/Chapters/MCMC_diagnostics.html
#  1. R statistics
#  2. E-BFMI https://python.arviz.org/projects/stats/en/stable/api/generated/arviz_stats.bfmi.html
#  3. trace plots
#  4. rank plots
#  5. Divergent transitions
#  6. Low effective sample size (ESS)
# see also arviz_stats.diagnose

In [ ]:
q = np.percentile(m_samples, [5, 25, 50, 75, 95], axis=0)
plt.plot(x, np.exp(q[2]), color="C0", label="MCMC inferred rate")
plt.fill_between(x, np.exp(q[0]), np.exp(q[-1]), alpha=0.3, lw=0, color="C0")
plt.fill_between(x, np.exp(q[1]), np.exp(q[-2]), alpha=0.3, lw=0, color="C0")
plt.plot(x, np.exp(true_log_rate), "--", color="C1", label="true rate")
plt.plot(x, y, ".k", label="data")
plt.legend(loc=2)
plt.xlabel("x")
_ = plt.ylabel("counts")

## Numpyro version

In [ ]:
%%time

import matplotlib.pyplot as plt
import numpy as np
import jax
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist

from tinygp import GaussianProcess, kernels

jax.config.update("jax_enable_x64", True)
numpyro.set_host_device_count(4)

random = np.random.default_rng(203618)
x = np.linspace(-3, 3, 20)
true_log_rate = 2 * np.cos(2 * x)
y = random.poisson(np.exp(true_log_rate))


def model(x, y=None):
    # The parameters of the GP model
    # mean = numpyro.sample("mean", dist.Normal(20, 0.01))
    mean = numpyro.sample("mean", dist.Normal(0, 2))
    sigma = numpyro.sample("sigma", dist.HalfNormal(3.0))
    rho = numpyro.sample("rho", dist.HalfNormal(10.0))

    # Set up the kernel and GP objects
    kernel = sigma**2 * kernels.Matern52(rho)
    gp = GaussianProcess(kernel, x, diag=1e-5, mean=mean)

    # This parameter has shape (num_data,) and it encodes our beliefs about
    # the process rate in each bin
    log_rate = numpyro.sample("log_rate", gp.numpyro_dist())

    # Finally, our observation model is Poisson
    numpyro.sample("obs", dist.Poisson(jnp.exp(log_rate)), obs=y)


# Run the MCMC
nuts_kernel = numpyro.infer.NUTS(model, target_accept_prob=0.9)
mcmc = numpyro.infer.MCMC(
    nuts_kernel,
    num_warmup=1000,
    num_samples=1000,
    num_chains=2,
    progress_bar=False,
)
rng_key = jax.random.PRNGKey(55873)
mcmc.run(rng_key, x, y=y)
samples = mcmc.get_samples()

q = np.percentile(samples["log_rate"], [5, 25, 50, 75, 95], axis=0)
plt.plot(x, np.exp(q[2]), color="C0", label="MCMC inferred rate")
plt.fill_between(x, np.exp(q[0]), np.exp(q[-1]), alpha=0.3, lw=0, color="C0")
plt.fill_between(x, np.exp(q[1]), np.exp(q[-2]), alpha=0.3, lw=0, color="C0")
plt.plot(x, np.exp(true_log_rate), "--", color="C1", label="true rate")
plt.plot(x, y, ".k", label="data")
plt.legend(loc=2)
plt.xlabel("x")
_ = plt.ylabel("counts")

In [ ]:
print(f"Total elasped time for the notebook: {(datetime.datetime.now() - start)}")